In [57]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, average_precision_score

In [58]:
df = pd.read_csv('creditcard.csv')

In [59]:
n_estimators_options = [50, 100, 300, 500]
max_depth_options    = [5, 10, 20, None]

In [60]:
y  = df['Class']
df.drop(columns="Class", inplace=True)
X_all = df

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=42
)

results = []

for n_estimators in n_estimators_options:
    for max_depth in max_depth_options:
        print(f"Training n_estimators={n_estimators}, max_depth={max_depth}...")
        
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            class_weight='balanced',
            random_state=42,
            min_samples_leaf=15
        )

        model.fit(X_train, y_train)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        results.append({
            'n_estimators': n_estimators,
            'max_depth':    max_depth,
            'recall':       recall_score(y_test, y_pred),
            'precision':    precision_score(y_test, y_pred),
            'f1':           f1_score(y_test, y_pred),
            'pr_auc':       average_precision_score(y_test, y_proba),
        })

results_df = pd.DataFrame(results).sort_values('pr_auc', ascending=False)
print(results_df.to_string(index=False))
results_df.to_csv('new_grid_search_results.csv', index=False)

Training n_estimators=50, max_depth=5...


KeyboardInterrupt: 

In [ ]:
new_results = pd.read_csv('new_grid_search_results.csv')

In [ ]:
metrics = ['recall', 'precision', 'f1', 'pr_auc']

for metric in metrics:
    best = new_results.sort_values(metric, ascending=False).iloc[0]
    
    print(f"\n══════════════════════════════════════════════════════")
    print(f"BEST MODEL BY {metric.upper()}")
    print(f"══════════════════════════════════════════════════════")
    print(f"  Recall:       {best['recall']:.4f}")
    print(f"  Precision:    {best['precision']:.4f}")
    print(f"  F1:           {best['f1']:.4f}")
    print(f"  PR-AUC:       {best['pr_auc']:.4f}")
    print(f"  n_estimators: {best['n_estimators']}")
    print(f"  max_depth:    {best['max_depth']}")


══════════════════════════════════════════════════════
BEST MODEL BY RECALL
══════════════════════════════════════════════════════
  Recall:       0.8878
  Precision:    0.3283
  F1:           0.4793
  PR-AUC:       0.7436
  n_estimators: 50.0
  max_depth:    5.0

══════════════════════════════════════════════════════
BEST MODEL BY PRECISION
══════════════════════════════════════════════════════
  Recall:       0.8265
  Precision:    0.7941
  F1:           0.8100
  PR-AUC:       0.8265
  n_estimators: 50.0
  max_depth:    20.0

══════════════════════════════════════════════════════
BEST MODEL BY F1
══════════════════════════════════════════════════════
  Recall:       0.8265
  Precision:    0.7941
  F1:           0.8100
  PR-AUC:       0.8265
  n_estimators: 50.0
  max_depth:    20.0

══════════════════════════════════════════════════════
BEST MODEL BY PR_AUC
══════════════════════════════════════════════════════
  Recall:       0.8163
  Precision:    0.7921
  F1:           0.8040
  P

In [ ]:
tuned = pd.read_csv('tuning_results_not_used.csv')

In [ ]:
metrics = ['recall', 'precision', 'f1']

for metric in metrics:
    best = tuned.sort_values(metric, ascending=False).iloc[0]
    
    print(f"\n══════════════════════════════════════════════════════")
    print(f"BEST MODEL BY {metric.upper()}")
    print(f"══════════════════════════════════════════════════════")
    print(f"  Recall:       {best['recall']:.4f}")
    print(f"  Precision:    {best['precision']:.4f}")
    print(f"  F1:           {best['f1']:.4f}")
    print(f"  n_estimators: {best['n_estimators']}")
    print(f"  max_depth:    {best['max_depth']}")
    print(f"  Features used: {best['features']}")



══════════════════════════════════════════════════════
BEST MODEL BY RECALL
══════════════════════════════════════════════════════
  Recall:       0.8878
  Precision:    0.1590
  F1:           0.2698
  n_estimators: 50
  max_depth:    5.0
  Features used: ['V14', 'V12', 'V2', 'V1', 'V3', 'V5']

══════════════════════════════════════════════════════
BEST MODEL BY PRECISION
══════════════════════════════════════════════════════
  Recall:       0.8061
  Precision:    0.9634
  F1:           0.8778
  n_estimators: 500
  max_depth:    nan
  Features used: ['V14', 'V12', 'V2', 'V1', 'V3', 'V5', 'V6', 'V8', 'V17', 'V11', 'V16', 'V10', 'V20']

══════════════════════════════════════════════════════
BEST MODEL BY F1
══════════════════════════════════════════════════════
  Recall:       0.8061
  Precision:    0.9634
  F1:           0.8778
  n_estimators: 500
  max_depth:    nan
  Features used: ['V14', 'V12', 'V2', 'V1', 'V3', 'V5', 'V6', 'V8', 'V17', 'V11', 'V16', 'V10', 'V20']


In [63]:
from sklearn.model_selection import cross_validate
model = RandomForestClassifier(n_estimators=50, max_depth=5, class_weight='balanced', random_state=42, min_samples_leaf=15)

scores = cross_validate(model, X_all, y, cv=5, scoring=['recall', 'precision', 'f1', 'average_precision'])
print(pd.DataFrame(scores).mean())

fit_time                  215.829922
score_time                  0.183069
test_recall                 0.843311
test_precision              0.435225
test_f1                     0.547375
test_average_precision      0.737764
dtype: float64


In [67]:
model = RandomForestClassifier(n_estimators=50, max_depth=None, class_weight='balanced', random_state=42, min_samples_leaf=15)
print(X_all.columns.tolist())
scores = cross_validate(model, X_all, y, cv=5, scoring=['recall', 'precision', 'f1', 'average_precision'])
print(pd.DataFrame(scores).mean())

['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
fit_time                  51.609484
score_time                 0.243876
test_recall                0.796434
test_precision             0.827987
test_f1                    0.805234
test_average_precision     0.759793
dtype: float64


In [56]:
model = RandomForestClassifier(n_estimators=500, max_depth=None, class_weight='balanced', random_state=42, min_samples_leaf=15)
feauters = ['V14', 'V12', 'V2', 'V1', 'V3', 'V5', 'V6', 'V8', 'V17', 'V11', 'V16', 'V10', 'V20']
X_all = df[feauters]
scores = cross_validate(model, X_all, y, cv=5, scoring=['recall', 'precision', 'f1', 'average_precision'])
print(pd.DataFrame(scores).mean())

fit_time                  256.577791
score_time                  1.872203
test_recall                 0.808658
test_precision              0.825026
test_f1                     0.811777
test_average_precision      0.746143
dtype: float64


In [55]:
model = RandomForestClassifier(n_estimators=50, max_depth=5, class_weight='balanced', random_state=42, min_samples_leaf=15)
features = ['V14', 'V12', 'V2', 'V1', 'V3', 'V5']
X_all = df[features]

scores = cross_validate(model, X_all, y, cv=5, scoring=['recall', 'precision', 'f1', 'average_precision'])
print(pd.DataFrame(scores).mean())

fit_time                  166.783331
score_time                  1.960007
test_recall                 0.794414
test_precision              0.773758
test_f1                     0.770408
test_average_precision      0.736149
dtype: float64


In [ ]:
new_df = pd.read_csv("best_from_the_best.csv")

metrics = ['recall', 'precision', 'f1', 'pr_auc']

for metric in metrics:
    best = tuned.sort_values(metric, ascending=False).iloc[0]
    
    print(f"\n══════════════════════════════════════════════════════")
    print(f"BEST MODEL BY {metric.upper()}")
    print(f"══════════════════════════════════════════════════════")
    print(f"  Recall:       {best['recall']:.4f}")
    print(f"  Precision:    {best['precision']:.4f}")
    print(f"  F1:           {best['f1']:.4f}")
    print(f"  n_estimators: {best['n_estimators']}")
    print(f"  max_depth:    {best['max_depth']}")
    print(f"  Features used: {best['features']}")